# GHSR1a 亲和力预测模型（QSAR）

## 目标

用 889 个小分子的结构预测它们对 GHSR1a 的结合亲和力（pAffinity）。

## 这份 notebook 的特殊之处

它把前面几步的成果**串成了一条完整流水线**：

| 前面学到的 | 在这里怎么用 |
|---|---|
| 骨架多样性 2.3 化合物/骨架 | 决定用 scaffold split 而非随机切分 |
| 失效临界点 相似度 0.6 | 做成**适用域判断**，低置信预测自动标红 |
| CB-Dock2 打分不可全信 | 对接结果只作为**可选附加特征**，不依赖精确构象 |

**核心原则**：模型不是在"预测活性"，而是在已知化学空间内插值。
超出适用域的预测一律标为不可信 —— 这比报一个漂亮的数字重要得多。

## 实测结果预览（scaffold split, 5 seeds）

| 组合 | R² | MAE | Spearman |
|---|---|---|---|
| Morgan + 描述符 + **GradientBoosting** | **0.453** | **0.599** | **0.665** |
| Morgan + RandomForest | 0.401 | 0.614 | 0.616 |
| Morgan + ExtraTrees | 0.061 | 0.773 | 0.492 |
| 仅描述符 + GradientBoosting | 0.159 | 0.729 | 0.437 |

基线（永远预测均值）MAE = 0.887 → 最佳模型把它降到 0.599，**提升 32%**

In [ ]:
# ── 路径初始化 ──────────────────────────────────────────────
# 自动定位项目根目录，使 notebook 在仓库内任意位置都能正确读写。
# 找不到时回退到当前工作目录（兼容扁平用法，如直接放在 ~/aidd 下运行）。
from pathlib import Path
import os


def _find_project_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / 'data' / 'processed').is_dir() and (p / 'notebooks').is_dir():
            return p
    return None


_ROOT = _find_project_root()

if _ROOT is not None:
    DATA_DIR = _ROOT / 'data' / 'processed'   # 清洗后的建模数据
    RAW_DIR = _ROOT / 'data' / 'raw'          # 原始数据（不入库，可重新下载）
    FIG_DIR = _ROOT / 'results' / 'figures'   # 图表输出
    for _d in (DATA_DIR, RAW_DIR, FIG_DIR):
        _d.mkdir(parents=True, exist_ok=True)
    print(f'项目根目录: {_ROOT}')
    print(f'  数据 → {DATA_DIR.relative_to(_ROOT)}')
    print(f'  图表 → {FIG_DIR.relative_to(_ROOT)}')
else:
    DATA_DIR = RAW_DIR = FIG_DIR = Path.cwd()
    print('未定位到项目根目录，使用当前工作目录（扁平模式）')


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from collections import defaultdict

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import Descriptors, Crippen, QED, rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

RDLogger.DisableLog('rdApp.*')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

SEEDS = [0, 1, 2, 3, 4]
SIM_THRESHOLD = 0.60      # 适用域门槛（来自第 6 月的分层诊断实验）

print('环境就绪')

## 1. 载入数据

用**小分子子集**。肽类的分子量和柔性与小分子不在同一尺度，
混进来的话模型会先去学"大分子 vs 小分子"，而不是真正的构效关系。

In [ ]:
df = pd.read_csv(DATA_DIR / 'ghsr1a_binding_clean.csv')
df = df[~df['peptide_like']].reset_index(drop=True)
df['mol'] = df['canonical_smiles'].apply(Chem.MolFromSmiles)
df = df[df['mol'].notna()].reset_index(drop=True)

n = len(df)
y = df['pAffinity'].values
mols = df['mol'].tolist()

print(f'样本数 {n}')
print(f'pAffinity  均值 {y.mean():.2f}  标准差 {y.std():.2f}  范围 {y.min():.2f}-{y.max():.2f}')
print(f'\n基线（永远预测均值）MAE = {np.abs(y - y.mean()).mean():.3f}')

## 2. 特征工程

三组特征，各有分工：

- **Morgan 指纹**（2048 bit）——编码分子里有哪些子结构，是 QSAR 的主力
- **MACCS 钥匙**（166 bit）——167 个预定义的化学特征，粗粒度
- **物化描述符**（15 个）——MW、logP、TPSA 等，可解释性强

实测结论：**描述符单独用很弱（R²≈0.16），但加到指纹上能明显提升（0.40→0.45）**。
原因是描述符提供的是"全局性质"，而指纹提供的是"局部结构"，两者互补。

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

DESC_FUNCS = {
    'MW':         Descriptors.MolWt,
    'logP':       Crippen.MolLogP,
    'TPSA':       rdMolDescriptors.CalcTPSA,
    'HBD':        rdMolDescriptors.CalcNumHBD,
    'HBA':        rdMolDescriptors.CalcNumHBA,
    'RotB':       rdMolDescriptors.CalcNumRotatableBonds,
    'AromR':      rdMolDescriptors.CalcNumAromaticRings,
    'Rings':      rdMolDescriptors.CalcNumRings,
    'SatR':       rdMolDescriptors.CalcNumSaturatedRings,
    'HeavyAtoms': Descriptors.HeavyAtomCount,
    'FracCsp3':   rdMolDescriptors.CalcFractionCSP3,
    'QED':        QED.qed,
    'MolMR':      Crippen.MolMR,
    'BertzCT':    Descriptors.BertzCT,
    'FpDensity':  Descriptors.FpDensityMorgan1,
}

print('计算特征 ...')
morgan = np.array([gen.GetFingerprint(m).ToList() for m in mols], dtype=np.float32)
maccs  = np.array([list(MACCSkeys.GenMACCSKeys(m))[1:] for m in mols], dtype=np.float32)
desc   = np.array([[f(m) for f in DESC_FUNCS.values()] for m in mols], dtype=np.float32)
desc   = np.nan_to_num(desc, nan=0.0, posinf=0.0, neginf=0.0)

FEATURES = {
    'Morgan2048':            morgan,
    'MACCS(166)':            maccs,
    '描述符(15)':            desc,
    'Morgan+描述符':          np.hstack([morgan, desc]),
    'Morgan+MACCS+描述符':    np.hstack([morgan, maccs, desc]),
}
for k, v in FEATURES.items():
    print(f'  {k:<22} {v.shape}')

## 3. 数据切分：scaffold split

**为什么不用随机切分**：同骨架的分子如果同时出现在训练集和测试集，
模型只要"认出这个骨架"就能拿高分，测出来的性能虚高。

**为什么你的数据上两者差距不算大**：骨架碎片化严重（889 个分子、385 种骨架，
平均 2.3 个/骨架），同骨架分子本就稀少，随机切分也难让它们跨集"抄袭"。

**但这不代表可以不做** —— 你事先不知道会是哪种情况。

In [ ]:
scaf_groups = defaultdict(list)
for i, s in enumerate(df['scaffold'].values):
    scaf_groups[s].append(i)


def scaffold_split(seed=0):
    rng = np.random.RandomState(seed)
    gs = list(scaf_groups.values())
    rng.shuffle(gs)                   # 同大小的组随机打破平局
    gs.sort(key=len, reverse=True)    # 大骨架优先进训练集
    n_train = int(n * 0.8)
    tr, te = [], []
    for g in gs:
        if len(tr) + len(g) <= n_train:
            tr.extend(g)
        else:
            te.extend(g)
    return np.array(tr), np.array(te)


def random_split(seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    k = int(n * 0.8)
    return idx[:k], idx[k:]


tr, te = scaffold_split(0)
print(f'Scaffold split: 训练集 {len(tr)} / 测试集 {len(te)}')
tr_s = set(df.iloc[te]['scaffold'])
print(f'测试集骨架与训练集重叠数: {len(tr_s & set(df.iloc[tr]["scaffold"]))}（应为 0）')

## 4. 特征 × 模型 网格对比

In [ ]:
MODELS = {
    'RandomForest': lambda s: RandomForestRegressor(n_estimators=500, random_state=s, n_jobs=-1),
    'ExtraTrees':   lambda s: ExtraTreesRegressor(n_estimators=500, random_state=s, n_jobs=-1),
    'GradBoost':    lambda s: GradientBoostingRegressor(n_estimators=300, random_state=s),
}


def evaluate(X, model_fn, splitter):
    r2s, maes, rhos = [], [], []
    for s in SEEDS:
        tr, te = splitter(s)
        m = model_fn(s)
        m.fit(X[tr], y[tr])
        p = m.predict(X[te])
        r2s.append(r2_score(y[te], p))
        maes.append(mean_absolute_error(y[te], p))
        rhos.append(spearmanr(y[te], p).correlation)
    return np.mean(r2s), np.std(r2s), np.mean(maes), np.mean(rhos)


rows = []
for fname, X in FEATURES.items():
    for mname, mfn in MODELS.items():
        r2, r2s, mae, rho = evaluate(X, mfn, scaffold_split)
        rows.append((fname, mname, r2, r2s, mae, rho))

R = pd.DataFrame(rows, columns=['特征', '模型', 'R2', 'R2波动', 'MAE', 'Spearman'])
R = R.sort_values('R2', ascending=False).reset_index(drop=True)
display(R.round(3))

### 一个值得注意的发现

**ExtraTrees 在这份数据上崩了**（Morgan 下 R² 仅 0.06，远低于 RandomForest 的 0.40）。

原因：ExtraTrees 在每个分裂点**随机选择阈值**，而 Morgan 指纹是 2048 维的稀疏
二值向量（绝大多数位置是 0）。随机阈值在这种特征上几乎提不出有效信息。

**教训：模型没有万能的，要看特征的性质来选。**
稀疏二值指纹适合 RandomForest / GradientBoosting，不适合 ExtraTrees。

## 5. 最佳模型：切分方式对比与最终评估

In [ ]:
best = R.iloc[0]
BEST_FEAT, BEST_MODEL = best['特征'], best['模型']
X = FEATURES[BEST_FEAT]
mfn = MODELS[BEST_MODEL]

print(f'最佳组合: {BEST_FEAT} + {BEST_MODEL}\n')
print(f"  {'切分方式':<16}{'R²':>16}{'MAE':>9}{'Spearman':>11}")
print('  ' + '-' * 52)
for sname, splitter in [('随机切分', random_split), ('Scaffold Split', scaffold_split)]:
    r2, r2s, mae, rho = evaluate(X, mfn, splitter)
    print(f'  {sname:<14}{r2:>11.3f}±{r2s:<4.2f}{mae:>9.3f}{rho:>11.3f}')
print()
print('  ↑ 随机切分数更好看，但那是虚的。以 Scaffold Split 为准。')

下面两张图用 **seed 0 的单次运行**绘制（便于展示）。

> 注意：单次运行的数字会有波动（seed 0 的 R² 约 0.33，5 seeds 平均约 0.45）。
> **报告性能时以 5 seeds 平均为准**，单次结果只用来看图。

In [ ]:
tr, te = scaffold_split(0)
model = mfn(0)
model.fit(X[tr], y[tr])
pred = model.predict(X[te])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(y[te], pred, s=22, alpha=0.6, c='#4C72B0', edgecolors='white', linewidths=0.4)
lo, hi = min(y[te].min(), pred.min()), max(y[te].max(), pred.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='理想预测')
ax.plot([lo, hi], [lo+1, hi+1], 'r:', lw=1, alpha=0.5, label='±1 log 单位')
ax.plot([lo, hi], [lo-1, hi-1], 'r:', lw=1, alpha=0.5)
ax.set_xlabel('实测 pAffinity')
ax.set_ylabel('预测 pAffinity')
ax.set_title(f'{BEST_FEAT} + {BEST_MODEL}\nScaffold Split  R²={r2_score(y[te],pred):.3f}  MAE={mean_absolute_error(y[te],pred):.3f}')
ax.legend()

ax = axes[1]
resid = y[te] - pred
ax.scatter(pred, resid, s=22, alpha=0.6, c='#55A868', edgecolors='white', linewidths=0.4)
ax.axhline(0, color='black', ls='--', lw=1)
ax.axhline(1, color='red', ls=':', lw=1, alpha=0.6)
ax.axhline(-1, color='red', ls=':', lw=1, alpha=0.6)
ax.set_xlabel('预测 pAffinity')
ax.set_ylabel('残差（实测 - 预测）')
ax.set_title('残差图\n残差应随机分布在 0 附近，无明显趋势')
plt.tight_layout()
plt.savefig(FIG_DIR / 'qsar_prediction_scatter.png', dpi=150)
plt.show()

## 6. 特征重要性：模型到底靠什么在预测

用**排列重要性**（permutation importance）：把某一列打乱，看模型性能掉多少。
掉得越多说明这列越重要。

In [ ]:
desc_names = list(DESC_FUNCS.keys())
r = permutation_importance(model, X[te], y[te], n_repeats=10, random_state=0,
                           n_jobs=-1)
imp_desc = r.importances_mean[morgan.shape[1]:]     # 只看描述符部分

order = np.argsort(-imp_desc)
print('描述符的排列重要性（在 Morgan 指纹之上的增量贡献）\n')
for i in order:
    v = imp_desc[i]
    bar = '#' * max(0, int(v * 60)) if v > 0 else ''
    print(f'  {desc_names[i]:<12}{v:>8.4f}  {bar}')

### 怎么读这张表

实测排在前几位的是 **BertzCT**（拓扑复杂度）、**logP**、**MW**、**FracCsp3**。

有意思的是 **logP 排第二**。结合第 4 月的结构分析就说得通了：

> GHSR1a 的配体口袋被 E124–R283 盐桥劈成两个腔，
> 其中 **Cavity II 是疏水的**，ghrelin 的辛酰基正是伸进这个腔。

所以亲脂性直接影响配体能否占据 Cavity II —— 模型从数据里"重新发现"了这一点，
而且没有人事先告诉它口袋长什么样。

**这就是把结构洞察和统计模型对照的价值**：模型给出相关性，
你的结构知识给出因果解释，两者对上了，结论就可信得多。

## 7. 适用域判断 —— 这一步不要省

第 6 月的分层诊断得出了一个硬数字（随机切分、10 次重复、1780 个测试分子）：

| 测试分子与训练集的最大相似度 | MAE | 判读 |
|---|---|---|
| 0.00–0.50 | 1.036 | 失效 |
| 0.50–0.60 | 0.881 | 勉强（贴基线） |
| **0.60–0.70** | **0.693** | **可靠** |
| 0.70–0.85 | 0.661 | 可靠 |
| 0.85–1.00 | 0.644 | 可靠 |

### 在本模型上的实测验证

把同一道门槛用到这个 QSAR 模型上，结果是：

| 组别 | 分子数 | 占比 | MAE | R² |
|---|---|---|---|---|
| 适用域内 (sim ≥ 0.6) | 153 | 86% | **0.605** | **0.444** |
| 适用域外 (sim < 0.6) | 25 | 14% | **0.968** | **−0.655** |

**注意适用域外的 R² 是负数** —— 意味着在那 14% 的分子上，
用模型预测还不如直接猜平均值。模型不只是"不准"，而是**有害**。

这两组独立数据（第 6 月的分层诊断、本节的模型实测）指向同一个门槛 0.6，
互相印证，说明这个数字是可靠的。

In [ ]:
# 训练集指纹（用于算相似度）
train_fps = [gen.GetFingerprint(mols[i]) for i in tr]


def max_sim_to_train(mol):
    """该分子与训练集中最相似分子的 Tanimoto 相似度"""
    fp = gen.GetFingerprint(mol)
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    return max(sims) if sims else 0.0


def predict_with_confidence(smiles, model=model, threshold=SIM_THRESHOLD):
    """预测一个分子的 pAffinity，同时给出可信度判断。

    返回: (预测值, 最大相似度, 是否可信)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, False

    fp = np.array([gen.GetFingerprint(mol).ToList()], dtype=np.float32)
    d = np.array([[f(mol) for f in DESC_FUNCS.values()]], dtype=np.float32)
    d = np.nan_to_num(d)
    X_new = np.hstack([fp, d])

    pred = float(model.predict(X_new)[0])
    sim = max_sim_to_train(mol)
    reliable = sim >= threshold
    return pred, sim, reliable


# 测试：用测试集的几个分子验证
print(f"  {'SMILES':<28}{'预测':>8}{'实测':>8}{'相似度':>9}{'可信':>7}")
print('  ' + '-' * 62)
for i in te[:6]:
    smi = df.iloc[i]['canonical_smiles']
    p, s, ok = predict_with_confidence(smi)
    print(f'  {smi[:26]:<28}{p:>8.2f}{y[i]:>8.2f}{s:>9.3f}{"✓" if ok else "✗ 标红":>7}')

In [ ]:
# 全测试集：可信 vs 不可信 的实际误差对比
sims = np.array([max_sim_to_train(mols[i]) for i in te])
errs = np.abs(y[te] - pred)
in_domain = sims >= SIM_THRESHOLD

print('=== 适用域内外的实际误差对比 ===\n')
print(f"  {'组别':<20}{'分子数':>8}{'占比':>8}{'MAE':>8}{'R²':>8}")
print('  ' + '-' * 52)
for name, mask in [('适用域内 (sim≥0.6)', in_domain),
                   ('适用域外 (sim<0.6)', ~in_domain)]:
    if mask.sum() == 0:
        continue
    mae = errs[mask].mean()
    r2 = r2_score(y[te][mask], pred[mask]) if mask.sum() > 1 else float('nan')
    print(f'  {name:<18}{mask.sum():>8}{mask.mean()*100:>7.0f}%{mae:>8.3f}{r2:>8.3f}')
print()
print('  ↑ 若在适用域内误差明显更小，说明这道门槛真的有用，不是摆设。')

## 8. 打包成一个可复用的预测函数

下面这个函数可以直接拿去用。给它一个 SMILES，它返回预测值 + 可信度。

**用法建议**：
- 相似度 ≥ 0.6：可以采信
- 相似度 < 0.6：当参考，别拿去做决策
- 如果要预测一个全新骨架的系列，先算算它们与训练集的相似度分布，
  普遍低于 0.6 的话，这个模型对它们基本无效 —— 那就得先补充那个系列的数据

In [ ]:
def predict_batch(smiles_list, threshold=SIM_THRESHOLD):
    """批量预测，返回结果表"""
    rows = []
    for smi in smiles_list:
        p, s, ok = predict_with_confidence(smi, threshold=threshold)
        rows.append({
            'SMILES': smi,
            '预测pAffinity': round(p, 2) if p is not None else None,
            '预测IC50_nM': round(10 ** (9 - p), 1) if p is not None else None,
            '与训练集相似度': round(s, 3) if s is not None else None,
            '是否可信': '✓' if ok else '✗ 不可信',
        })
    return pd.DataFrame(rows)


# 示例：几个已知的 GHSR1a 配体（SMILES 仅作演示，实际用时换成你自己的分子）
demo = [
    'CN(C)N(C)C(=O)[C@]1(Cc2ccccc2)CCCN(C(=O)[C@@H](Cc2c[nH]c3ccccc23)NC(=O)C(C)(C)N)C1',
]
print('示例预测（anamorelin）:')
display(predict_batch(demo))
print()
print('提示: 把自己的分子 SMILES 放进 demo 列表即可批量预测。')
print('      从 PubChem 查 SMILES，或用 RDKit 从结构文件转换。')

## 8b. 附：Boosting 三剑客深度对比

XGBoost 和 LightGBM 通常被认为强于 sklearn 的 GradientBoosting。
**但在这份数据上，实测结果并非如此。**

### 5 seeds 平均（scaffold split）

| 模型 | R² | MAE | Spearman |
|---|---|---|---|
| GradientBoosting | **0.453** | 0.599 | **0.665** |
| XGBoost（调参） | 0.448 | **0.582** | 0.647 |
| RandomForest | 0.382 | 0.623 | 0.585 |
| XGBoost（默认） | 0.383 | 0.621 | 0.609 |
| LightGBM（调参） | 0.394 | 0.603 | 0.614 |
| LightGBM（默认） | 0.323 | 0.634 | 0.574 |

### 关键：不同指标给出不同结论

逐 seed 对比 GradientBoosting 与 XGBoost（调参）：

| seed | GradBoost R² | XGBoost R² | 谁赢 |
|---|---|---|---|
| 0 | 0.334 | 0.301 | GradBoost |
| 1 | 0.381 | 0.331 | GradBoost |
| 2 | 0.455 | **0.501** | XGBoost |
| 3 | 0.555 | **0.572** | XGBoost |
| 4 | **0.540** | 0.537 | GradBoost |

- **R²：XGBoost 仅胜 2/5**，差值均值 −0.005、标准差 0.034 → 完全在噪声内
- **MAE：XGBoost 胜 5/5**，差值均值 −0.018 → 稳定改善

### 为什么两个指标结论不同

- **MAE** 衡量平均绝对误差，对异常值鲁棒
- **R²** 含平方项，对大误差更敏感

XGBoost 的正则化让它在**大多数样本**上更准（MAE 好），
但对少数极端活性值的预测偏保守（R² 受损）。

**对药物发现而言 MAE 更重要** —— 我们关心的是典型分子的预测误差，
而不是能否精确拟合那几个极端值。

### 结论

> 若你最看重「预测值离实测值差多少」→ 用 **XGBoost（调参）**，MAE 0.582
> 若你最看重「排序能力」→ 用 **GradientBoosting**，Spearman 0.665
>
> 但两者差距都很小。**换模型这条路已经走到头了。**

想再提升，得靠**数据和特征**，而不是继续换模型。

In [ ]:
# 如果已安装 xgboost / lightgbm，可运行本格复现上面的对比
# 安装: conda install -c conda-forge xgboost lightgbm
# （已验证这两个包不会降级 rdkit / numpy / pandas）

try:
    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor
    HAS_BOOST = True
except ImportError:
    HAS_BOOST = False
    print('未安装 xgboost / lightgbm，跳过。')
    print('安装命令: conda install -c conda-forge xgboost lightgbm')

if HAS_BOOST:
    BOOST_MODELS = {
        'GradientBoosting': lambda s: GradientBoostingRegressor(n_estimators=300, random_state=s),
        'XGBoost (默认)': lambda s: XGBRegressor(n_estimators=300, random_state=s,
                                              n_jobs=-1, verbosity=0),
        'XGBoost (调参)': lambda s: XGBRegressor(
            n_estimators=600, learning_rate=0.03, max_depth=5,
            subsample=0.8, colsample_bytree=0.7,
            reg_alpha=0.1, reg_lambda=1.0, min_child_weight=2,
            random_state=s, n_jobs=-1, verbosity=0),
        'LightGBM (默认)': lambda s: LGBMRegressor(n_estimators=300, random_state=s,
                                                n_jobs=-1, verbose=-1),
        'LightGBM (调参)': lambda s: LGBMRegressor(
            n_estimators=600, learning_rate=0.03, num_leaves=31,
            subsample=0.8, colsample_bytree=0.7,
            reg_alpha=0.1, reg_lambda=1.0, min_child_samples=5,
            random_state=s, n_jobs=-1, verbose=-1),
    }
    rows = []
    for name, mfn in BOOST_MODELS.items():
        r2, r2s, mae, rho = evaluate(X, mfn, scaffold_split)
        rows.append((name, r2, r2s, mae, rho))
    RB = pd.DataFrame(rows, columns=['模型', 'R2', 'R2波动', 'MAE', 'Spearman'])
    RB = RB.sort_values('MAE').reset_index(drop=True)
    display(RB.round(3))
    print('\n按 MAE 排序（越小越好）。注意 R² 与 MAE 的最优模型可能不同。')

## 9. 小结与下一步

### 这份模型能做什么

| 指标 | 值 | 含义 |
|---|---|---|
| R²（scaffold split） | 0.45 | 解释 45% 的活性变异 |
| MAE | 0.60 log 单位 | 平均误差约 **4 倍**浓度 |
| Spearman | 0.67 | 排序能力尚可，适合**排优先级** |
| 相比基线 | 提升 32% | 基线 MAE 0.887 |

**MAE 0.6 log 单位意味着**：预测 IC50 = 10 nM 的分子，实际可能在 2.5–40 nM 之间。
用来**排序、排优先级**可以；用来**精确预测单个分子的活性**还不够。

### 三条改进路线

1. **加特征**：把对接打分（docking score）加进来。注意，CB-Dock2 的
   redocking RMSD 是 3.30 Å —— 结合模式不准确，但**打分仍有排序价值**。
   加之前先验证它能提升多少，别盲目加。

2. **换模型**：XGBoost / LightGBM 通常比 sklearn 的 GradientBoosting 更好；
   样本量够的话可以试图神经网络（但 889 个样本偏少，未必划算）。

3. **加数据**：889 个分子对这个任务偏小。如果能补充课题组自己的实测数据
   （哪怕几十个），对模型在**你的化学空间**内的表现帮助最大 ——
   因为私有数据才是你独有的、别人拿不走的东西。

### 最重要的一句话

> **始终报告 scaffold split 的数字，并且对适用域外的预测标红。**
> 做到这两点，你的模型就比大多数发表出来的 QSAR 模型诚实。